# GR00T N1.7 VLA Lab: Mac data work → MuJoCo → Linux GPU fine-tuning → MCAP cleaning

**Updated for macOS and Linux/Docker workflows; public APIs/docs checked 2026-09-15.**

This notebook deliberately has two execution modes:

### Mode A — macOS / CPU development
You can run:
- GR00T / LeRobot dataset inspection;
- plotting and data QA;
- the standalone **MuJoCo fundamentals lab**;
- MCAP generation, decoding, synchronization and cleaning;
- LIBERO dataset download/inspection.

You **do not** run the current NVIDIA GR00T N1.7 CUDA training stack on macOS.

### Mode B — Linux + NVIDIA CUDA, including Docker
You can additionally run:
- GR00T N1.7 model loading;
- fine-tuning;
- open-loop inference;
- LIBERO closed-loop evaluation in MuJoCo;
- before/after checkpoint rollout videos.

The learning progression is:

```text
SO100 data inspection
        ↓
MuJoCo fundamentals
        ↓
GR00T fine-tune + training metrics
        ↓
open-loop action metrics
        ↓
LIBERO / MuJoCo closed-loop rollouts
        ↓
early checkpoint vs later checkpoint videos
        ↓
MCAP raw episode cleaning
        ↓
robot episode staged for validation
```

> **Important:** `LIBERO_PANDA` is a post-training embodiment in GR00T N1.7. The raw base model cannot be used directly as a LIBERO policy. Therefore the closed-loop "before vs after" experiment compares an **early fine-tuning checkpoint** with a **later/final checkpoint** using identical simulator seeds.

### Key public references

- GR00T: https://github.com/NVIDIA/Isaac-GR00T
- GR00T policy/inference guide: https://github.com/NVIDIA/Isaac-GR00T/blob/main/getting_started/policy.md
- GR00T fine-tuning: https://github.com/NVIDIA/Isaac-GR00T/blob/main/getting_started/finetune_new_embodiment.md
- GR00T LIBERO benchmark: https://github.com/NVIDIA/Isaac-GR00T/blob/main/examples/LIBERO/README.md
- MuJoCo Python: https://mujoco.readthedocs.io/en/stable/python.html
- LIBERO: https://github.com/Lifelong-Robot-Learning/LIBERO
- MCAP ROS 2: https://mcap.dev/guides/python/ros2

## 0. Platform expectations

### macOS

Use macOS for the **data + simulation learning parts** of this notebook.

The official `mujoco` Python package installs with:

```bash
pip install mujoco
```

and bundles the MuJoCo runtime. A special `mjpython` launcher is only needed for some interactive viewer patterns on macOS; this notebook uses normal stepping/offscreen rendering and falls back gracefully if rendering is unavailable.

The current GR00T N1.7 root environment is NVIDIA CUDA-oriented, so the notebook **does not call `uv sync` for GR00T on macOS**.

### Linux / Docker + NVIDIA GPU

For full GR00T training/evaluation you need:
- Linux;
- NVIDIA GPU with appropriate VRAM;
- NVIDIA driver / CUDA runtime;
- for Docker: NVIDIA Container Toolkit and `--gpus all`;
- EGL/OpenGL system libraries for headless MuJoCo/LIBERO rendering.

NVIDIA currently recommends about **40 GB+ GPU VRAM** for the standard GR00T N1.7 fine-tuning path.

A typical GPU-container pattern is:

```bash
docker run --rm -it \
  --gpus all \
  --ipc=host \
  -p 8888:8888 \
  -v "$PWD":/workspace \
  -w /workspace \
  <NVIDIA-PyTorch-or-CUDA-image>
```

Inside a Debian/Ubuntu-based container, make sure these are available before LIBERO evaluation:

```bash
apt-get update
apt-get install -y git git-lfs ffmpeg libegl1-mesa-dev libglu1-mesa
```

### Hugging Face access

GR00T N1.7 loads the gated `nvidia/Cosmos-Reason2-2B` backbone. For training/inference:
1. request/accept access on Hugging Face;
2. authenticate via `HF_TOKEN` or the Hugging Face CLI.

In [ ]:
import json
import math
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path(os.environ.get("GR00T_LAB_ROOT", Path.cwd() / "gr00t_lab")).resolve()
REPO_DIR = WORK_ROOT / "Isaac-GR00T"

SYSTEM = platform.system()
IS_MAC = SYSTEM == "Darwin"
IS_LINUX = SYSTEM == "Linux"
# Container runtimes expose the lowercase 'container' variable.
IN_DOCKER = Path("/.dockerenv").exists() or bool(os.environ.get("container"))  # noqa: SIM112


def command_ok(cmd):
    try:
        return (
            subprocess.run(
                cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=10
            ).returncode
            == 0
        )
    except Exception:
        return False


HAS_NVIDIA_SMI = shutil.which("nvidia-smi") is not None
HAS_VISIBLE_NVIDIA_GPU = HAS_NVIDIA_SMI and command_ok(["nvidia-smi", "-L"])
FULL_GROOT_SUPPORTED = IS_LINUX and HAS_VISIBLE_NVIDIA_GPU

# Heavy operations are opt-in.
RUN_SIMPLE_FINETUNE = False
RUN_SIMPLE_EVAL = False
RUN_COMPLEX_DOWNLOAD = False
RUN_COMPLEX_FINETUNE = False
RUN_LIBERO_SIM_SETUP = False
RUN_MUJOCO_BEFORE_AFTER = False

# A smoke test proves the pipeline; it does not produce a strong policy.
SIMPLE_MAX_STEPS = 50
COMPLEX_MAX_STEPS = 100
GLOBAL_BATCH_SIZE = 32
LIBERO_SUITE = "goal"  # goal, object, spatial, 10

WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("System              :", SYSTEM)
print("Docker              :", IN_DOCKER)
print("NVIDIA GPU visible  :", HAS_VISIBLE_NVIDIA_GPU)
print("Full GR00T mode     :", FULL_GROOT_SUPPORTED)
print("WORK_ROOT           :", WORK_ROOT)

if IS_MAC:
    print("\nMode: macOS data / MuJoCo / MCAP lab. CUDA GR00T cells will skip cleanly.")
elif FULL_GROOT_SUPPORTED:
    print("\nMode: Linux CUDA. Full GR00T training and LIBERO evaluation are available.")
else:
    print(
        "\nMode: Linux without visible NVIDIA GPU. Data/MuJoCo/MCAP cells work; GR00T training skips."
    )

## 1. Check the machine and rendering prerequisites

In [ ]:
usage = shutil.disk_usage(WORK_ROOT)
print("OS:", platform.platform())
print("Python kernel:", sys.version.split()[0])
print(f"Free disk: {usage.free / 2**30:.1f} GiB")


def show_command(cmd):
    if not shutil.which(cmd[0]):
        print(f"\n{cmd[0]}: not found")
        return
    p = subprocess.run(cmd, text=True, capture_output=True)
    print(f"\n$ {' '.join(cmd)}")
    print((p.stdout or p.stderr).strip()[:8000])


show_command(["nvidia-smi"])
show_command(["ffmpeg", "-version"])

if IN_DOCKER and FULL_GROOT_SUPPORTED:
    print("\nDocker GPU looks visible. LIBERO additionally needs EGL/OpenGL libraries.")
elif IN_DOCKER and not FULL_GROOT_SUPPORTED:
    print("\nDocker detected but NVIDIA GPU is not visible.")
    print("Start the container with NVIDIA Container Toolkit and --gpus all.")

### System packages

The notebook does not run OS package managers automatically.

**macOS** (Homebrew, if needed):

```bash
brew install git git-lfs ffmpeg
```

**Debian/Ubuntu Linux or container**:

```bash
apt-get update
apt-get install -y git git-lfs ffmpeg libegl1-mesa-dev libglu1-mesa
```

MuJoCo itself comes from the Python package; there is no separate MuJoCo binary download required for the normal Python API.

## 2. Install cross-platform notebook dependencies

In [ ]:
import sys

pkgs = [
    "numpy",
    "pandas",
    "pyarrow",
    "matplotlib",
    "pillow",
    "imageio",
    "imageio-ffmpeg",
    "huggingface_hub",
    "mujoco",
    "mcap",
    "mcap-ros2-support",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Cross-platform notebook dependencies installed.")

## 3. Clone GR00T; create the CUDA environment only where supported

Cloning is useful on **both** Mac and Linux because the repository contains the small SO100 demo dataset and the reference configs.

`uv sync` for the GR00T root environment is only attempted on **Linux with a visible NVIDIA GPU**.

In [ ]:
from pathlib import Path

if not REPO_DIR.exists():
    subprocess.check_call(
        [
            "git",
            "clone",
            "--recurse-submodules",
            "https://github.com/NVIDIA/Isaac-GR00T.git",
            str(REPO_DIR),
        ]
    )
else:
    print("Repository already exists:", REPO_DIR)

# Pull LFS objects when git-lfs is available.
if shutil.which("git-lfs") or command_ok(["git", "lfs", "version"]):
    try:
        subprocess.check_call(["git", "lfs", "pull"], cwd=REPO_DIR)
    except Exception as e:
        print("git lfs pull warning:", e)
else:
    print("git-lfs not found. Install it if demo media/parquet files appear as tiny pointer files.")

# uv is cross-platform; install it if absent, but only sync the GR00T CUDA env on supported Linux.
uv = shutil.which("uv")
if uv is None:
    try:
        subprocess.check_call(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"])
    except Exception as e:
        print("uv installer warning:", e)

    candidates = [
        shutil.which("uv"),
        str(Path.home() / ".local/bin/uv"),
        str(Path.home() / ".cargo/bin/uv"),
    ]
    uv = next((x for x in candidates if x and Path(x).exists()), None)

print("uv:", uv)

if FULL_GROOT_SUPPORTED:
    if uv is None:
        raise RuntimeError("uv is required for full GR00T mode.")
    subprocess.check_call([uv, "sync", "--python", "3.12"], cwd=REPO_DIR)
    subprocess.check_call(
        [uv, "run", "python", "-c", "import gr00t; print('GR00T import OK')"],
        cwd=REPO_DIR,
    )
else:
    print("Skipping GR00T `uv sync`: full model stack requires Linux + visible NVIDIA GPU.")

## 4. Hugging Face authentication check

In [ ]:
from huggingface_hub import HfApi

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    who = HfApi(token=HF_TOKEN).whoami()
    print("Hugging Face user:", who.get("name", who))
except Exception as e:
    print("Hugging Face auth not active:", type(e).__name__)
    print("Dataset downloads may still work when public; GR00T gated model access requires login.")

if FULL_GROOT_SUPPORTED and not HF_TOKEN:
    print("\nFor model training/inference you can run:")
    print("  huggingface-cli login")
    print("or export HF_TOKEN before starting Jupyter.")

# Part A — simple manipulation: SO100 cube → bowl

The bundled `cube_to_bowl_5` dataset exposes the GR00T data contract without a large download.

Its modality config uses front + wrist RGB, current robot state, **5 arm values + 1 gripper value**, a **16-step future action horizon**, relative arm actions, absolute gripper actions and language.

Five demonstrations are useful for learning the mechanics, not for claiming a strong policy.

In [ ]:
SIMPLE_DATASET = REPO_DIR / "demo_data" / "cube_to_bowl_5"
assert SIMPLE_DATASET.exists(), f"Missing {SIMPLE_DATASET}"
print(SIMPLE_DATASET)

for p in sorted(SIMPLE_DATASET.rglob("*")):
    if p.is_file():
        rel = p.relative_to(SIMPLE_DATASET)
        print(f"{rel!s:85s} {p.stat().st_size / 2**20:8.2f} MiB")

## 5. Inspect the metadata contract

`meta/modality.json` is the GR00T-specific bridge between flat arrays in parquet and meaningful robot fields.

In [ ]:
from pprint import pprint


def load_json(path):
    with open(path) as f:
        return json.load(f)


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


modality = load_json(SIMPLE_DATASET / "meta" / "modality.json")
info = load_json(SIMPLE_DATASET / "meta" / "info.json")
episodes = read_jsonl(SIMPLE_DATASET / "meta" / "episodes.jsonl")
tasks = read_jsonl(SIMPLE_DATASET / "meta" / "tasks.jsonl")

print("modality.json")
pprint(modality)
print("\nFirst episode metadata:")
pprint(episodes[0])
print("\nTasks:")
pprint(tasks[:10])

## 6. Inspect one episode at timestep level

A GR00T/LeRobot row is not an `(instruction, answer)` pair. It contains timestamped state, action, episode identity and task annotation. Video lives in parallel MP4 files.

In [ ]:
import numpy as np
import pandas as pd

simple_parquets = sorted(SIMPLE_DATASET.glob("data/chunk-*/episode_*.parquet"))
assert simple_parquets
ep0 = pd.read_parquet(simple_parquets[0])

display(ep0.head())
print("Columns:", list(ep0.columns))
print("Rows:", len(ep0))

state0 = np.asarray(ep0.iloc[0]["observation.state"], dtype=float)
action0 = np.asarray(ep0.iloc[0]["action"], dtype=float)
print("State dimension :", state0.shape)
print("Action dimension:", action0.shape)
print("First state     :", state0)
print("First action    :", action0)

## 7. Reusable dataset QA

Checks required columns, finite state/action data, timestamp order, gaps, jump norms and video counts.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

REQUIRED_COLUMNS = {
    "observation.state",
    "action",
    "timestamp",
    "episode_index",
    "index",
    "task_index",
    "next.done",
}


def _stack_array_col(series):
    return np.stack([np.asarray(x, dtype=float).reshape(-1) for x in series])


def robust_threshold(x, z=8.0):
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return np.inf
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if mad < 1e-12:
        return med + max(1e-6, np.nanstd(x) * 6)
    return med + z * 1.4826 * mad


def audit_gr00t_dataset(root: Path):
    root = Path(root)
    rows = []
    parquets = sorted(root.glob("data/chunk-*/episode_*.parquet"))

    for pq in parquets:
        df = pd.read_parquet(pq)
        missing = sorted(REQUIRED_COLUMNS - set(df.columns))
        if missing:
            rows.append({"episode": pq.name, "missing_columns": ",".join(missing)})
            continue

        ts = df["timestamp"].to_numpy(dtype=float)
        dt = np.diff(ts)
        states = _stack_array_col(df["observation.state"])
        actions = _stack_array_col(df["action"])
        state_jump = (
            np.linalg.norm(np.diff(states, axis=0), axis=1) if len(states) > 1 else np.array([])
        )
        action_jump = (
            np.linalg.norm(np.diff(actions, axis=0), axis=1) if len(actions) > 1 else np.array([])
        )
        positive_dt = dt[dt > 0]
        median_dt = np.median(positive_dt) if len(positive_dt) else np.nan
        gap_threshold = 3 * median_dt if np.isfinite(median_dt) else np.nan

        rows.append(
            {
                "episode": pq.name,
                "rows": len(df),
                "state_dim": states.shape[1],
                "action_dim": actions.shape[1],
                "duration_s": float(ts[-1] - ts[0]) if len(ts) else 0.0,
                "median_dt_s": float(median_dt) if np.isfinite(median_dt) else np.nan,
                "nonmonotonic_or_duplicate_ts": int(np.sum(dt <= 0)),
                "large_gaps": int(np.sum(dt > gap_threshold)) if np.isfinite(gap_threshold) else 0,
                "nonfinite_state_values": int((~np.isfinite(states)).sum()),
                "nonfinite_action_values": int((~np.isfinite(actions)).sum()),
                "state_jump_flags": int(np.sum(state_jump > robust_threshold(state_jump))),
                "action_jump_flags": int(np.sum(action_jump > robust_threshold(action_jump))),
                "done_on_last_row": bool(df.iloc[-1]["next.done"]) if len(df) else False,
                "missing_columns": "",
            }
        )

    result = pd.DataFrame(rows)
    videos = sorted(root.glob("videos/chunk-*/*/episode_*.mp4"))
    camera_dirs = sorted({v.parent.name for v in videos})
    print(f"Episodes: {len(parquets)}")
    print(f"Videos  : {len(videos)} across cameras {camera_dirs}")
    return result


simple_audit = audit_gr00t_dataset(SIMPLE_DATASET)
display(simple_audit)

## 8. Plot state and action traces

In [ ]:
import matplotlib.pyplot as plt

ts = ep0["timestamp"].to_numpy(float)
states = _stack_array_col(ep0["observation.state"])
actions = _stack_array_col(ep0["action"])

plt.figure(figsize=(12, 5))
for i in range(states.shape[1]):
    plt.plot(ts, states[:, i], label=f"state[{i}]")
plt.xlabel("time (s)")
plt.ylabel("state")
plt.title("SO100 episode 0 — observation.state")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 5))
for i in range(actions.shape[1]):
    plt.plot(ts, actions[:, i], label=f"action[{i}]")
plt.xlabel("time (s)")
plt.ylabel("action")
plt.title("SO100 episode 0 — action")
plt.legend(ncol=3)
plt.show()

## 9. View the camera episode

In [ ]:
from IPython.display import Video, display

simple_videos = sorted(SIMPLE_DATASET.glob("videos/chunk-*/*/episode_000000.mp4"))
for video in simple_videos:
    print(video.relative_to(SIMPLE_DATASET))
    display(Video(str(video), embed=False, width=480))

## 10. Read the actual SO100 modality config

Separate **dataset metadata** (`meta/modality.json`) from the **model sampling config** (`so100_config.py`). Changing the action horizon requires recomputing statistics.

In [ ]:
so100_cfg = REPO_DIR / "examples" / "SO100" / "so100_config.py"
print(so100_cfg.read_text())

## 11. Fine-tune the simple task

NVIDIA's documented single-GPU example uses 2000 steps and global batch size 32. This notebook defaults to **50 steps** only to prove the pipeline works.

Set `RUN_SIMPLE_FINETUNE = True` on a 40 GB+ VRAM GPU.

In [ ]:
SIMPLE_OUT = WORK_ROOT / "checkpoints" / "so100_cube_to_bowl"
SIMPLE_OUT.mkdir(parents=True, exist_ok=True)

simple_cmd = [
    uv,
    "run",
    "python",
    "gr00t/experiment/launch_finetune.py",
    "--base-model-path",
    "nvidia/GR00T-N1.7-3B",
    "--dataset-path",
    str(SIMPLE_DATASET),
    "--embodiment-tag",
    "NEW_EMBODIMENT",
    "--modality-config-path",
    "examples/SO100/so100_config.py",
    "--num-gpus",
    "1",
    "--output-dir",
    str(SIMPLE_OUT),
    "--max-steps",
    str(SIMPLE_MAX_STEPS),
    "--save-steps",
    str(max(1, SIMPLE_MAX_STEPS)),
    "--save-total-limit",
    "2",
    "--global-batch-size",
    str(GLOBAL_BATCH_SIZE),
    "--dataloader-num-workers",
    "4",
]

print("Command:\n", " ".join(map(str, simple_cmd)))

if RUN_SIMPLE_FINETUNE and FULL_GROOT_SUPPORTED:
    subprocess.check_call(simple_cmd, cwd=REPO_DIR)
elif RUN_SIMPLE_FINETUNE and not FULL_GROOT_SUPPORTED:
    print("Requested training, but this runtime is not Linux + NVIDIA CUDA.")
else:
    print("\nNot executed. Set RUN_SIMPLE_FINETUNE=True on Linux + NVIDIA GPU.")

## 12. Optional open-loop evaluation

Open-loop evaluation compares predicted action chunks with recorded ground truth. It is useful for debugging but does **not** prove closed-loop task success.

In [ ]:
def newest_checkpoint(folder: Path):
    ckpts = [p for p in Path(folder).glob("checkpoint-*") if p.is_dir()]

    def step(p):
        m = re.search(r"checkpoint-(\d+)$", p.name)
        return int(m.group(1)) if m else -1

    return max(ckpts, key=step) if ckpts else None


simple_ckpt = newest_checkpoint(SIMPLE_OUT)
print("Newest checkpoint:", simple_ckpt)

if RUN_SIMPLE_EVAL and FULL_GROOT_SUPPORTED:
    if simple_ckpt is None:
        raise RuntimeError("No simple checkpoint found. Run fine-tuning first.")
    eval_cmd = [
        uv,
        "run",
        "python",
        "gr00t/eval/open_loop_eval.py",
        "--dataset-path",
        str(SIMPLE_DATASET),
        "--embodiment-tag",
        "NEW_EMBODIMENT",
        "--model-path",
        str(simple_ckpt),
        "--traj-ids",
        "0",
        "--execution-horizon",
        "16",
        "--steps",
        "200",
    ]
    print(" ".join(map(str, eval_cmd)))
    subprocess.check_call(eval_cmd, cwd=REPO_DIR)
elif RUN_SIMPLE_EVAL and not FULL_GROOT_SUPPORTED:
    print("Requested GR00T inference, but this runtime is not Linux + NVIDIA CUDA.")
else:
    print("Set RUN_SIMPLE_EVAL=True after training to run open-loop evaluation.")

# Training and inference metrics

For this VLA lab, separate four questions:

| Question | Metric |
|---|---|
| Is optimization progressing? | training loss, learning rate, gradient norm |
| Does the action predictor match demonstrations? | open-loop action MSE / MAE |
| Which control dimensions are wrong? | per-action-dimension traces/errors |
| Does the robot complete the task? | closed-loop MuJoCo success rate |

A conventional `eval_loss` is possible in GR00T's trainer when a real evaluation dataset is configured and `eval_strategy` is enabled. Do **not** manufacture a frame-level random split of robot trajectories just to get an `eval_loss`: adjacent frames leak heavily.

The bundled five-episode SO100 demo is too small for a statistically meaningful validation split. We therefore emphasize:
- local training-loss curves;
- episode-level open-loop action metrics;
- closed-loop LIBERO success rate.

If `eval_loss` exists in `trainer_state.json`, the plotting code below will show it automatically.

In [ ]:
def load_trainer_history(output_dir: Path):
    output_dir = Path(output_dir)
    ckpts = sorted(
        [p for p in output_dir.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1,
    )
    if not ckpts:
        return pd.DataFrame()

    state_path = ckpts[-1] / "trainer_state.json"
    if not state_path.exists():
        print("No trainer_state.json:", state_path)
        return pd.DataFrame()

    state = json.loads(state_path.read_text())
    hist = pd.DataFrame(state.get("log_history", []))
    print("Loaded:", state_path)
    print("Global step:", state.get("global_step"))
    return hist


def plot_training_history(output_dir: Path, title="GR00T training"):
    hist = load_trainer_history(output_dir)
    if hist.empty:
        print("No metrics yet.")
        return hist

    display(hist.tail(20))

    metric_groups = [
        ["loss", "eval_loss"],
        ["learning_rate"],
        ["grad_norm"],
        ["train_accuracy", "eval_accuracy"],
    ]

    for metrics in metric_groups:
        available = [m for m in metrics if m in hist.columns and hist[m].notna().any()]
        if not available:
            continue

        plt.figure(figsize=(10, 4))
        for metric in available:
            subset = hist[["step", metric]].dropna()
            plt.plot(subset["step"], subset[metric], marker=".", label=metric)
        plt.xlabel("training step")
        plt.ylabel("metric")
        plt.title(f"{title}: {', '.join(available)}")
        plt.legend()
        plt.show()

    return hist


# After running SO100 fine-tuning:
simple_history = plot_training_history(SIMPLE_OUT, "SO100 cube → bowl")

### Open-loop checkpoint comparison

GR00T's `open_loop_eval.py` reports unnormalized **MSE** and **MAE** and saves plots comparing predicted and demonstrated actions.

For a useful learning experiment, save multiple checkpoints and evaluate the **same episode IDs** against each checkpoint.

Example:

```text
checkpoint-250     MSE 0.12
checkpoint-1000    MSE 0.05
checkpoint-2000    MSE 0.03
```

Then check whether those improvements also produce higher **closed-loop success**. The two metrics can diverge.

In [ ]:
def list_checkpoints(output_dir: Path):
    def step(p):
        m = re.fullmatch(r"checkpoint-(\d+)", p.name)
        return int(m.group(1)) if m else -1

    return sorted(
        [
            p
            for p in Path(output_dir).glob("checkpoint-*")
            if p.is_dir() and re.fullmatch(r"checkpoint-\d+", p.name)
        ],
        key=step,
    )


def checkpoint_save_interval(max_steps):
    if isinstance(max_steps, bool) or not isinstance(max_steps, int) or max_steps < 2:
        raise ValueError("Use at least 2 training steps to save an early and a later checkpoint.")
    return max_steps // 2


def validate_checkpoint_pair(early, late):
    if early is None or late is None:
        raise ValueError(
            "Two checkpoints are required. Fine-tune with the configured save interval "
            "or select two existing checkpoints."
        )
    early, late = Path(early).resolve(), Path(late).resolve()
    if not early.is_dir() or not late.is_dir():
        raise ValueError("Both checkpoint directories must exist before comparing them.")
    if early == late:
        raise ValueError("Early and late must be different checkpoints, not the same directory.")
    return early, late


print("SO100 checkpoints:")
for p in list_checkpoints(SIMPLE_OUT):
    print(" ", p.name)

print("""
For each checkpoint, run open_loop_eval.py on identical episode IDs and save a separate plot.
The official evaluator prints:
  - per-trajectory MSE
  - per-trajectory MAE
  - average MSE / MAE
and draws ground-truth vs predicted action traces.
""")

# Part B — MuJoCo fundamentals (works on Mac and Linux)

Before using LIBERO, spend a few minutes with MuJoCo directly.

MuJoCo has two central objects:

```text
MjModel  = static model: bodies, joints, geometry, actuators, timestep
MjData   = changing simulation state: qpos, qvel, controls, contacts, time
```

The essential loop is:

```python
model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)

data.ctrl[:] = ...
mujoco.mj_step(model, data)
```

This section uses a small actuated 2-link arm so you can see:
- MJCF;
- joints and actuators;
- control input;
- physics stepping;
- `qpos` / `qvel`;
- optional camera rendering.

It does **not** depend on GR00T or CUDA.

In [ ]:
import matplotlib.pyplot as plt
import mujoco
import numpy as np

MJCF = r"""
<mujoco model="two_link_learning_arm">
  <option timestep="0.002" gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <geom type="plane" size="2 2 0.1" rgba="0.9 0.9 0.9 1"/>

    <body name="base" pos="0 0 0.15">
      <geom type="cylinder" size="0.12 0.15" rgba="0.2 0.2 0.2 1"/>
      <body name="link1" pos="0 0 0.15">
        <joint name="shoulder" type="hinge" axis="0 1 0" range="-120 120" damping="1"/>
        <geom name="link1_geom" type="capsule" fromto="0 0 0 0 0 0.7" size="0.05"
              rgba="0.2 0.4 0.8 1"/>
        <body name="link2" pos="0 0 0.7">
          <joint name="elbow" type="hinge" axis="0 1 0" range="-140 140" damping="1"/>
          <geom name="link2_geom" type="capsule" fromto="0 0 0 0 0 0.55" size="0.045"
                rgba="0.8 0.4 0.2 1"/>
          <site name="eef" pos="0 0 0.55" size="0.04" rgba="0 1 0 1"/>
        </body>
      </body>
    </body>
  </worldbody>

  <actuator>
    <position name="shoulder_servo" joint="shoulder" kp="60" ctrlrange="-1.6 1.6"/>
    <position name="elbow_servo" joint="elbow" kp="45" ctrlrange="-1.8 1.8"/>
  </actuator>
</mujoco>
"""

mj_model = mujoco.MjModel.from_xml_string(MJCF)
mj_data = mujoco.MjData(mj_model)

print("MuJoCo version :", mujoco.__version__)
print("nq / nv / nu   :", mj_model.nq, mj_model.nv, mj_model.nu)
print("timestep       :", mj_model.opt.timestep)

In [ ]:
# Command two joint-position targets and record the response.
duration_s = 3.0
target = np.array([0.8, -1.0], dtype=float)

times, qpos_log, qvel_log = [], [], []

while mj_data.time < duration_s:
    mj_data.ctrl[:] = target
    mujoco.mj_step(mj_model, mj_data)

    if len(times) == 0 or mj_data.time - times[-1] >= 0.01:
        times.append(mj_data.time)
        qpos_log.append(mj_data.qpos.copy())
        qvel_log.append(mj_data.qvel.copy())

qpos_log = np.asarray(qpos_log)
qvel_log = np.asarray(qvel_log)

plt.figure(figsize=(10, 4))
plt.plot(times, qpos_log[:, 0], label="shoulder qpos")
plt.plot(times, qpos_log[:, 1], label="elbow qpos")
plt.axhline(target[0], linestyle="--", label="shoulder target")
plt.axhline(target[1], linestyle="--", label="elbow target")
plt.xlabel("simulation time (s)")
plt.ylabel("joint position (rad)")
plt.title("MuJoCo: position actuators driving a 2-link arm")
plt.legend()
plt.show()

print("Final qpos:", mj_data.qpos.copy())
print("Final qvel:", mj_data.qvel.copy())

In [ ]:
# Optional offscreen render.
# Rendering depends on a working graphics backend. Physics stepping above is independent of rendering.

try:
    renderer = mujoco.Renderer(mj_model, height=360, width=480)
    renderer.update_scene(mj_data)
    rgb = renderer.render()

    plt.figure(figsize=(8, 6))
    plt.imshow(rgb)
    plt.axis("off")
    plt.title("MuJoCo offscreen render")
    plt.show()
    renderer.close()
except Exception as e:
    print("Offscreen rendering unavailable in this runtime:", repr(e))
    if IS_MAC:
        print(
            "On macOS, interactive viewer workflows may require `mjpython`; physics/data cells still work."
        )
    if IN_DOCKER:
        print("In Docker, check EGL/OpenGL libraries and NVIDIA GPU passthrough.")

### What changes when we move from this toy arm to LIBERO?

Almost nothing about the **simulation loop**:

```text
toy MuJoCo                      LIBERO / MuJoCo
-----------                     ----------------
mj_step()                       env.step(action)
qpos/qvel                       Panda proprioception
simple camera                   task cameras
2 joint controls                7-DoF Panda + gripper
manual target                   GR00T action chunk
no task condition               language-conditioned task
```

LIBERO adds robot/task assets, object scenes, observations, task-reset logic and success conditions around MuJoCo.

# Part C — more complex manipulation: LIBERO Panda

LIBERO adds a Franka/Panda-style setting, multiple tasks, language-conditioned behavior, richer scene goals and closed-loop simulation.

This notebook defaults to **LIBERO Goal**. Change `LIBERO_SUITE = "10"` for longer multi-step tasks.
The evaluation task is selected from the same suite automatically. Dataset downloads, fine-tuning output, published checkpoint selection and rollout commands all use `LIBERO_SUITE`.


In [ ]:
LIBERO_REPOS = {
    "goal": "IPEC-COMMUNITY/libero_goal_no_noops_1.0.0_lerobot",
    "object": "IPEC-COMMUNITY/libero_object_no_noops_1.0.0_lerobot",
    "spatial": "IPEC-COMMUNITY/libero_spatial_no_noops_1.0.0_lerobot",
    "10": "IPEC-COMMUNITY/libero_10_no_noops_1.0.0_lerobot",
}
# One example from each suite in NVIDIA's LIBERO task list.
LIBERO_DEFAULT_ENVS = {
    "goal": "libero_sim/put_the_bowl_on_the_plate",
    "object": "libero_sim/pick_up_the_alphabet_soup_and_place_it_in_the_basket",
    "spatial": "libero_sim/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate",
    "10": "libero_sim/KITCHEN_SCENE3_turn_on_the_stove_and_put_the_moka_pot_on_it",
}


def libero_env_for_suite(suite):
    if suite not in LIBERO_DEFAULT_ENVS:
        raise ValueError(f"Unknown LIBERO suite {suite!r}; choose goal, object, spatial or 10.")
    return LIBERO_DEFAULT_ENVS[suite]


LIBERO_ENV = libero_env_for_suite(LIBERO_SUITE)

repo_id = LIBERO_REPOS[LIBERO_SUITE]
suite_dir_name = repo_id.split("/")[-1]
LIBERO_DATASET = REPO_DIR / "examples" / "LIBERO" / suite_dir_name

print("Suite:", LIBERO_SUITE)
print("Evaluation task:", LIBERO_ENV)
print("HF dataset:", repo_id)
print("Local path:", LIBERO_DATASET)

## 13. Download LIBERO

The official GR00T recipe downloads a LeRobot dataset from Hugging Face and copies NVIDIA's `modality.json` into `meta/`.

For LIBERO Goal, NVIDIA also ships a replacement for one known corrupted wrist-camera episode.

In [ ]:
from huggingface_hub import snapshot_download

print("Dataset:", repo_id)
print("Destination:", LIBERO_DATASET)

if RUN_COMPLEX_DOWNLOAD:
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=str(LIBERO_DATASET),
        token=HF_TOKEN,
    )

    meta_dir = LIBERO_DATASET / "meta"
    meta_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO_DIR / "examples/LIBERO/modality.json", meta_dir / "modality.json")

    if LIBERO_SUITE == "goal":
        patch = REPO_DIR / "examples/LIBERO/patches/episode_000082.mp4"
        target = (
            LIBERO_DATASET / "videos/chunk-000/observation.images.wrist_image/episode_000082.mp4"
        )
        if patch.exists():
            shutil.copy2(patch, target)
            print("Applied NVIDIA wrist-video patch:", target)
else:
    print("Not executed. Set RUN_COMPLEX_DOWNLOAD=True.")
    print("This dataset download works on both macOS and Linux.")

## 14. Audit LIBERO

In [ ]:
if LIBERO_DATASET.exists():
    libero_audit = audit_gr00t_dataset(LIBERO_DATASET)
    display(libero_audit.head(20))
    libero_tasks = read_jsonl(LIBERO_DATASET / "meta/tasks.jsonl")
    print("\nSample task descriptions:")
    for row in libero_tasks[:20]:
        print(row)
else:
    print("LIBERO dataset not downloaded yet.")

## 15. Fine-tune LIBERO — learning run versus reference run

NVIDIA's published recipe uses about 20,000 steps, 8 GPUs in the reference command, global batch size 640 and `state_dropout_prob=0.2`.

The one-GPU 100-step setting below is only for learning the mechanics.
The notebook saves halfway through the run and at subsequent intervals, retaining two checkpoints. With the default 100 steps this produces checkpoints at steps 50 and 100. At least two training steps are required.


In [ ]:
COMPLEX_OUT = WORK_ROOT / "checkpoints" / f"libero_{LIBERO_SUITE}"
COMPLEX_OUT.mkdir(parents=True, exist_ok=True)

complex_cmd = [
    uv,
    "run",
    "python",
    "gr00t/experiment/launch_finetune.py",
    "--base-model-path",
    "nvidia/GR00T-N1.7-3B",
    "--dataset-path",
    str(LIBERO_DATASET),
    "--embodiment-tag",
    "LIBERO_PANDA",
    "--num-gpus",
    "1",
    "--output-dir",
    str(COMPLEX_OUT),
    "--max-steps",
    str(COMPLEX_MAX_STEPS),
    "--save-steps",
    str(checkpoint_save_interval(COMPLEX_MAX_STEPS)),
    "--save-total-limit",
    "2",
    "--global-batch-size",
    str(GLOBAL_BATCH_SIZE),
    "--dataloader-num-workers",
    "4",
    "--state-dropout-prob",
    "0.2",
]

print("Learning/smoke-test command:\n", " ".join(map(str, complex_cmd)))

if RUN_COMPLEX_FINETUNE and FULL_GROOT_SUPPORTED:
    if not LIBERO_DATASET.exists():
        raise RuntimeError("Download LIBERO first.")
    subprocess.check_call(complex_cmd, cwd=REPO_DIR)
elif RUN_COMPLEX_FINETUNE and not FULL_GROOT_SUPPORTED:
    print("Requested fine-tuning, but this runtime is not Linux + NVIDIA CUDA.")
else:
    print("\nNot executed. Set RUN_COMPLEX_FINETUNE=True on Linux + NVIDIA GPU.")

print("\nReference-scale NVIDIA recipe uses NUM_GPUS=8, MAX_STEPS=20000, GLOBAL_BATCH_SIZE=640.")

## 16. Download NVIDIA's already-fine-tuned LIBERO checkpoint

If your goal is to learn evaluation rather than pay for a full benchmark fine-tune, use NVIDIA's published checkpoint.

In [ ]:
PRETRAINED_LIBERO_ROOT = WORK_ROOT / "checkpoints" / "GR00T-N1.7-LIBERO"
checkpoint_subdir = f"libero_{LIBERO_SUITE}"

allow_patterns = [
    f"{checkpoint_subdir}/config.json",
    f"{checkpoint_subdir}/embodiment_id.json",
    f"{checkpoint_subdir}/model-*.safetensors",
    f"{checkpoint_subdir}/model.safetensors.index.json",
    f"{checkpoint_subdir}/processor_config.json",
    f"{checkpoint_subdir}/statistics.json",
]

print("Hugging Face model: nvidia/GR00T-N1.7-LIBERO")
print("Subset:", checkpoint_subdir)
print("Destination:", PRETRAINED_LIBERO_ROOT)
print("\nTo download, run:")
print("""
snapshot_download(
    repo_id="nvidia/GR00T-N1.7-LIBERO",
    local_dir=str(PRETRAINED_LIBERO_ROOT),
    allow_patterns=allow_patterns,
    token=HF_TOKEN,
)
""")

## 17. Closed-loop LIBERO evaluation = GR00T policy + MuJoCo environment

This is where MuJoCo becomes the **robot evaluation world** rather than a toy simulator.

The NVIDIA path is:

```text
GR00T policy server (CUDA)
        │
        │ actions
        ▼
LIBERO client
        │
        ▼
MuJoCo Panda + objects
        │
        ├── RGB observations
        ├── robot state
        └── success / failure
```

The current NVIDIA LIBERO setup script is Linux-oriented and configures **EGL** headless rendering. Therefore:

- **Mac:** run the standalone MuJoCo lab and data work locally; use Linux GPU/Docker for GR00T+LIBERO.
- **Linux GPU/Docker:** full closed-loop LIBERO evaluation is supported.

LIBERO uses sparse task success: a rollout matters because the robot actually completes the manipulation, not because its action MSE is low.

In [ ]:
setup_cmd = ["bash", "gr00t/eval/sim/LIBERO/setup_libero.sh"]
print("One-time LIBERO setup:", " ".join(setup_cmd))

if RUN_LIBERO_SIM_SETUP and FULL_GROOT_SUPPORTED:
    subprocess.check_call(setup_cmd, cwd=REPO_DIR)
elif RUN_LIBERO_SIM_SETUP and not FULL_GROOT_SUPPORTED:
    print("LIBERO+GR00T closed-loop setup requires Linux + NVIDIA CUDA.")
else:
    print("Not executed. Set RUN_LIBERO_SIM_SETUP=True on the Linux GPU machine.")

print(f"""
Official server-client pattern:

Terminal 1 — policy server:
  uv run python gr00t/eval/run_gr00t_server.py \\
    --model-path <LIBERO_CHECKPOINT> \\
    --embodiment-tag LIBERO_PANDA \\
    --use-sim-policy-wrapper

Terminal 2 — MuJoCo/LIBERO rollout:
  gr00t/eval/sim/LIBERO/libero_uv/.venv/bin/python gr00t/eval/rollout_policy.py \\
    --n-episodes 10 \\
    --policy-client-host 127.0.0.1 \\
    --policy-client-port 5555 \\
    --max-episode-steps 720 \\
    --env-name {LIBERO_ENV} \\
    --n-action-steps 8 \\
    --n-envs 1 \\
    --seed 42 \\
    --video-dir /tmp/libero_rollout_video
""")

## 18. Before/after inference in MuJoCo

For LIBERO, **do not compare the raw GR00T base model to a LIBERO checkpoint**. `LIBERO_PANDA` is not a base-model embodiment.

Instead compare:

```text
same MuJoCo task
same random seed
same episode count
same action horizon

EARLY checkpoint  ──► rollout videos + success rate
LATE checkpoint   ──► rollout videos + success rate
```

The default learning run saves at steps 50 and 100. Comparison requires two existing, distinct checkpoint directories; a missing checkpoint or two paths resolving to the same directory blocks the comparison.

The rollout evaluator can save videos with `--video-dir`, and it prints the final success rate.

In [ ]:
LIBERO_ENV = libero_env_for_suite(LIBERO_SUITE)
LIBERO_EPISODES = 10
LIBERO_SEED = 42

complex_ckpts = list_checkpoints(COMPLEX_OUT)
EARLY_CKPT = complex_ckpts[0] if complex_ckpts else None
LATE_CKPT = complex_ckpts[-1] if len(complex_ckpts) >= 2 else None

# You can also set LATE_CKPT to NVIDIA's published fully trained suite checkpoint:
# LATE_CKPT = PRETRAINED_LIBERO_ROOT / f"libero_{LIBERO_SUITE}"


def rollout_client_command(video_dir, port=5555):
    libero_python = REPO_DIR / "gr00t/eval/sim/LIBERO/libero_uv/.venv/bin/python"
    return [
        str(libero_python),
        "gr00t/eval/rollout_policy.py",
        "--n-episodes",
        str(LIBERO_EPISODES),
        "--policy-client-host",
        "127.0.0.1",
        "--policy-client-port",
        str(port),
        "--max-episode-steps",
        "720",
        "--env-name",
        LIBERO_ENV,
        "--n-action-steps",
        "8",
        "--n-envs",
        "1",
        "--seed",
        str(LIBERO_SEED),
        "--video-dir",
        str(video_dir),
    ]


COMPARISON_PAIR = None
try:
    COMPARISON_PAIR = validate_checkpoint_pair(EARLY_CKPT, LATE_CKPT)
except ValueError as error:
    if RUN_MUJOCO_BEFORE_AFTER:
        raise
    print("Comparison not ready:", error)

if COMPARISON_PAIR is not None:
    EARLY_CKPT, LATE_CKPT = COMPARISON_PAIR
    print("Early checkpoint:", EARLY_CKPT)
    print("Late checkpoint :", LATE_CKPT)
    if RUN_MUJOCO_BEFORE_AFTER:
        print("Run these commands from", REPO_DIR, "for each checkpoint, one at a time.")
        for label, checkpoint in zip(("early", "late"), COMPARISON_PAIR, strict=True):
            server = [
                uv or "uv",
                "run",
                "python",
                "gr00t/eval/run_gr00t_server.py",
                "--model-path",
                str(checkpoint),
                "--embodiment-tag",
                "LIBERO_PANDA",
                "--use-sim-policy-wrapper",
                "--host",
                "127.0.0.1",
                "--port",
                "5555",
            ]
            video_dir = WORK_ROOT / "videos" / f"libero_{LIBERO_SUITE}_{label}"
            print(f"\n{label.title()} — terminal 1 (stop the previous server first):")
            print(shlex.join(server))
            print(f"{label.title()} — terminal 2:")
            print(shlex.join(rollout_client_command(video_dir)))
    else:
        print("Set RUN_MUJOCO_BEFORE_AFTER=True to generate the two rollout command pairs.")

### Run both checkpoints

Set `RUN_MUJOCO_BEFORE_AFTER=True` and rerun the preceding cell to generate commands for both checkpoints. The flag generates commands; it does not start GPU processes automatically. For each checkpoint:

1. start the GR00T policy server with that checkpoint;
2. run the MuJoCo client with the **same seed**;
3. write videos to a separate directory;
4. save stdout to a log file.

Example server:

```bash
uv run python gr00t/eval/run_gr00t_server.py   --model-path /path/to/checkpoint-500   --embodiment-tag LIBERO_PANDA   --use-sim-policy-wrapper   --host 127.0.0.1   --port 5555
```

The client already prints:

```text
results: ...
success rate: 0.7
Video saved to: ...
```

Run again for the late checkpoint using the same environment and `--seed 42`.

In [ ]:
def parse_rollout_log(text):
    rate = None
    m = re.search(r"success rate:\s*([0-9.eE+-]+)", text)
    if m:
        rate = float(m.group(1))

    video = None
    m = re.search(r"Video saved to:\s*(.+)", text)
    if m:
        video = m.group(1).strip()

    return {"success_rate": rate, "video_dir": video}


def compare_success_rates(early_log_text, late_log_text):
    validate_checkpoint_pair(EARLY_CKPT, LATE_CKPT)
    early = parse_rollout_log(early_log_text)
    late = parse_rollout_log(late_log_text)

    frame = pd.DataFrame(
        [
            {"checkpoint": "early", "success_rate": early["success_rate"]},
            {"checkpoint": "late", "success_rate": late["success_rate"]},
        ]
    )
    display(frame)

    if frame["success_rate"].notna().all():
        plt.figure(figsize=(6, 4))
        plt.bar(frame["checkpoint"], frame["success_rate"])
        plt.ylim(0, 1)
        plt.ylabel("closed-loop task success rate")
        plt.title(f"MuJoCo / LIBERO before vs after — seed {LIBERO_SEED}")
        plt.show()

    return frame


# Example after saving logs:
# early_text = Path("early_rollout.log").read_text()
# late_text  = Path("late_rollout.log").read_text()
# compare_success_rates(early_text, late_text)

### What to look for in the videos

Do not only ask "did it succeed?"

Watch for:
- whether it reaches the correct object;
- grasp approach/orientation;
- oscillation;
- overshoot;
- whether the gripper closes at the right phase;
- recovery after small errors;
- repeated action patterns;
- collisions;
- whether failure occurs because of perception, planning or low-level action execution.

That qualitative failure analysis is often more informative than another decimal place of MSE.

# Part D — MCAP data cleaning exercise

This is the part that matters when demonstrations come from an industrial arm through ROS 2.

Typical raw rates differ:

```text
joint state       50–500 Hz
action/command    20–100 Hz
RGB camera        15–60 Hz
force/torque      100–1000 Hz
```

Do **not** concatenate messages by row number.

Use:

```text
ROS 2 / MCAP episode
        ↓
topic inventory
        ↓
timestamp QA
        ↓
remove duplicates / invalid messages
        ↓
detect sensor gaps + state/action jumps
        ↓
choose a common training timebase
        ↓
interpolate/nearest-sample each modality
        ↓
trim idle head/tail
        ↓
manual replay / task-success filter
        ↓
GR00T LeRobot episode
```

## 18. Create a deliberately dirty MCAP episode

The synthetic recording contains robot state, action, two camera-frame streams and task text. It intentionally has duplicate timestamps, an out-of-order timestamp, a camera gap and one implausible robot-state jump.

In [ ]:
from pathlib import Path

import numpy as np
from mcap_ros2.writer import Writer as McapWriter

MCAP_DIR = WORK_ROOT / "mcap"
MCAP_DIR.mkdir(parents=True, exist_ok=True)
SYNTH_MCAP = MCAP_DIR / "dirty_pick_place_episode.mcap"

STATE_SCHEMA = """\
float64[5] arm
float64 gripper
"""
ACTION_SCHEMA = """\
float64[5] arm
float64 gripper
"""
FRAME_SCHEMA = """\
uint32 frame_id
"""
TASK_SCHEMA = """\
string text
"""

base_ns = 1_700_000_000_000_000_000

with open(SYNTH_MCAP, "wb") as f:
    w = McapWriter(f)
    state_schema = w.register_msgdef("demo_msgs/RobotState", STATE_SCHEMA)
    action_schema = w.register_msgdef("demo_msgs/RobotAction", ACTION_SCHEMA)
    frame_schema = w.register_msgdef("demo_msgs/FrameMeta", FRAME_SCHEMA)
    task_schema = w.register_msgdef("demo_msgs/Task", TASK_SCHEMA)

    w.write_message(
        topic="/task",
        schema=task_schema,
        message={"text": "pick the red cube and place it in the bowl"},
        log_time=base_ns,
        publish_time=base_ns,
        sequence=0,
    )

    n_state = 300
    t_state = np.arange(n_state) / 50.0
    ts_state = base_ns + (t_state * 1e9).astype(np.int64)
    ts_state[65] = ts_state[64]
    ts_state[155] = ts_state[154] - 5_000_000

    for i, (t, ts) in enumerate(zip(t_state, ts_state, strict=False)):
        arm = np.array(
            [
                0.30 * np.sin(0.7 * t),
                0.25 * np.sin(0.9 * t + 0.3),
                0.20 * np.cos(0.6 * t),
                0.15 * np.sin(1.2 * t),
                0.10 * np.cos(0.8 * t),
            ]
        )
        if i == 190:
            arm = arm + 3.0
        gripper = 1.0 if t < 2.4 else 0.0
        w.write_message(
            topic="/robot/state",
            schema=state_schema,
            message={"arm": arm.tolist(), "gripper": float(gripper)},
            log_time=int(ts),
            publish_time=int(ts),
            sequence=i,
        )

    n_action = 120
    t_action = np.arange(n_action) / 20.0
    ts_action = base_ns + (t_action * 1e9).astype(np.int64)
    ts_action[48] = ts_action[47]
    for i, (t, ts) in enumerate(zip(t_action, ts_action, strict=False)):
        arm = np.array(
            [
                0.30 * np.sin(0.7 * (t + 0.08)),
                0.25 * np.sin(0.9 * (t + 0.08) + 0.3),
                0.20 * np.cos(0.6 * (t + 0.08)),
                0.15 * np.sin(1.2 * (t + 0.08)),
                0.10 * np.cos(0.8 * (t + 0.08)),
            ]
        )
        gripper = 1.0 if t < 2.3 else 0.0
        w.write_message(
            topic="/robot/action",
            schema=action_schema,
            message={"arm": arm.tolist(), "gripper": float(gripper)},
            log_time=int(ts),
            publish_time=int(ts),
            sequence=i,
        )

    frame_times = np.arange(90) / 15.0
    front_times = [t for t in frame_times if not (3.0 < t < 3.45)]
    for i, t in enumerate(front_times):
        ts = base_ns + int(t * 1e9)
        w.write_message(
            topic="/camera/front/frame",
            schema=frame_schema,
            message={"frame_id": i},
            log_time=ts,
            publish_time=ts,
            sequence=i,
        )

    for i, t in enumerate(frame_times):
        ts = base_ns + int(t * 1e9)
        if i == 40:
            ts = base_ns + int(frame_times[39] * 1e9)
        w.write_message(
            topic="/camera/wrist/frame",
            schema=frame_schema,
            message={"frame_id": i},
            log_time=ts,
            publish_time=ts,
            sequence=i,
        )

    w.finish()

print("Created:", SYNTH_MCAP)
print("Size:", f"{SYNTH_MCAP.stat().st_size / 1024:.1f} KiB")

## 19. Generic MCAP topic inventory — use this on real bags too

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd
from mcap.reader import make_reader


def inventory_mcap(path):
    rows = defaultdict(list)
    schemas = {}

    with open(path, "rb") as f:
        reader = make_reader(f)
        for schema, channel, message in reader.iter_messages(log_time_order=False):
            rows[channel.topic].append(int(message.log_time))
            schemas[channel.topic] = schema.name if schema else None

    out = []
    for topic, ts_ns in rows.items():
        ts = np.asarray(ts_ns, dtype=np.int64) / 1e9
        dt_record_order = np.diff(ts)
        ts_sorted = np.sort(ts)
        dt_sorted = np.diff(ts_sorted)
        positive = dt_sorted[dt_sorted > 0]
        med_dt = np.median(positive) if len(positive) else np.nan

        out.append(
            {
                "topic": topic,
                "schema": schemas[topic],
                "messages": len(ts),
                "span_s": float(ts_sorted[-1] - ts_sorted[0]) if len(ts) > 1 else 0.0,
                "median_rate_hz": float(1 / med_dt)
                if np.isfinite(med_dt) and med_dt > 0
                else np.nan,
                "duplicates": int(len(ts_sorted) - len(np.unique(ts_sorted))),
                "record_order_nonmonotonic": int(np.sum(dt_record_order < 0)),
                "gaps_gt_3x_median": int(np.sum(dt_sorted > 3 * med_dt))
                if np.isfinite(med_dt)
                else 0,
                "max_gap_s": float(np.max(dt_sorted)) if len(dt_sorted) else 0.0,
            }
        )

    return pd.DataFrame(out).sort_values("topic").reset_index(drop=True)


MCAP_PATH = SYNTH_MCAP
display(inventory_mcap(MCAP_PATH))

For a real recording:

```python
MCAP_PATH = Path("/data/episodes/episode_000123.mcap")
display(inventory_mcap(MCAP_PATH))
```

Look for inconsistent camera FPS, controller pauses, duplicates, cross-camera drift, force/torque saturation and missing TF.

## 20. Decode ROS 2 fields without installing ROS 2

In [ ]:
from mcap_ros2.decoder import DecoderFactory

with open(MCAP_PATH, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    seen = set()

    for schema, channel, _message, ros_msg in reader.iter_decoded_messages(log_time_order=False):
        if channel.topic not in seen:
            print("\nTOPIC:", channel.topic)
            print("TYPE :", schema.name)
            print("MSG  :", ros_msg)
            seen.add(channel.topic)
        if len(seen) >= 5:
            break

## 21. Extract robot streams

Customize this mapping for your robot. Keep source timestamps.

In [ ]:
def decode_demo_mcap(path):
    state_rows, action_rows, front_rows, wrist_rows, task_rows = [], [], [], [], []

    with open(path, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])
        for _schema, channel, message, ros_msg in reader.iter_decoded_messages(
            log_time_order=False
        ):
            t = message.log_time / 1e9
            topic = channel.topic

            if topic == "/robot/state":
                vals = list(ros_msg.arm)
                state_rows.append(
                    {
                        "t": t,
                        **{f"arm_{i}": float(v) for i, v in enumerate(vals)},
                        "gripper": float(ros_msg.gripper),
                    }
                )
            elif topic == "/robot/action":
                vals = list(ros_msg.arm)
                action_rows.append(
                    {
                        "t": t,
                        **{f"arm_{i}": float(v) for i, v in enumerate(vals)},
                        "gripper": float(ros_msg.gripper),
                    }
                )
            elif topic == "/camera/front/frame":
                front_rows.append({"t": t, "frame_id": int(ros_msg.frame_id)})
            elif topic == "/camera/wrist/frame":
                wrist_rows.append({"t": t, "frame_id": int(ros_msg.frame_id)})
            elif topic == "/task":
                task_rows.append({"t": t, "text": str(ros_msg.text)})

    return {
        "state": pd.DataFrame(state_rows),
        "action": pd.DataFrame(action_rows),
        "front": pd.DataFrame(front_rows),
        "wrist": pd.DataFrame(wrist_rows),
        "task": pd.DataFrame(task_rows),
    }


streams = decode_demo_mcap(MCAP_PATH)
for name, df in streams.items():
    print(name, df.shape)
    display(df.head(3))

## 22. Timestamp cleaning

Preserve raw records, sort, remove exact duplicates, report changes and never silently interpolate across a large outage.

In [ ]:
def clean_timestamps(df, name, gap_factor=3.0):
    x = df.copy()
    before = len(x)
    original_nonmono = int(np.sum(np.diff(x["t"].to_numpy()) <= 0)) if len(x) > 1 else 0

    x = x.sort_values("t").reset_index(drop=True)
    dup = int(x["t"].duplicated(keep="first").sum())
    x = x.drop_duplicates("t", keep="first").reset_index(drop=True)

    dt = np.diff(x["t"].to_numpy())
    pos = dt[dt > 0]
    med = np.median(pos) if len(pos) else np.nan
    gap_limit = gap_factor * med if np.isfinite(med) else np.nan
    gap_idx = np.where(dt > gap_limit)[0] if np.isfinite(gap_limit) else np.array([], dtype=int)

    report = {
        "stream": name,
        "rows_before": before,
        "rows_after_dedupe": len(x),
        "nonmonotonic_or_duplicate_in_record_order": original_nonmono,
        "duplicates_removed": dup,
        "median_dt_s": med,
        "gap_limit_s": gap_limit,
        "large_gap_count": int(len(gap_idx)),
        "large_gaps_after_row": gap_idx.tolist(),
    }
    return x, report


cleaned = {}
reports = []
for name in ["state", "action", "front", "wrist"]:
    cleaned[name], rep = clean_timestamps(streams[name], name)
    reports.append(rep)

display(pd.DataFrame(reports))

## 23. Detect implausible state jumps

For real hardware, replace the generic robust threshold with real joint-position, velocity, acceleration and EEF limits.

In [ ]:
state = cleaned["state"].copy()
joint_cols = [c for c in state.columns if c.startswith("arm_")]

dt = np.diff(state["t"].to_numpy())
dq = np.diff(state[joint_cols].to_numpy(), axis=0)
speed_norm = np.linalg.norm(dq / dt[:, None], axis=1)

speed_limit = robust_threshold(speed_norm, z=8.0)
bad_after = np.where(speed_norm > speed_limit)[0] + 1

print("Robust speed-norm threshold:", speed_limit)
print("Flagged row indices:", bad_after.tolist())

plt.figure(figsize=(12, 4))
plt.plot(state["t"].iloc[1:], speed_norm)
plt.axhline(speed_limit, linestyle="--", label="robust threshold")
plt.xlabel("time (s)")
plt.ylabel("joint-speed vector norm")
plt.title("State jump detection")
plt.legend()
plt.show()

## 24. Remove isolated bad samples with an audit trail

The demo removes a one-frame spike only if neighboring samples are mutually consistent. Long suspicious regions should trigger manual review.

In [ ]:
def remove_isolated_state_outliers(df, joint_cols, threshold):
    x = df.copy().reset_index(drop=True)
    drop = []
    arr = x[joint_cols].to_numpy(float)
    t = x["t"].to_numpy(float)

    for i in range(1, len(x) - 1):
        dt_prev = t[i] - t[i - 1]
        dt_next = t[i + 1] - t[i]
        dt_bridge = t[i + 1] - t[i - 1]
        if min(dt_prev, dt_next, dt_bridge) <= 0:
            continue

        v_prev = np.linalg.norm((arr[i] - arr[i - 1]) / dt_prev)
        v_next = np.linalg.norm((arr[i + 1] - arr[i]) / dt_next)
        v_bridge = np.linalg.norm((arr[i + 1] - arr[i - 1]) / dt_bridge)

        if v_prev > threshold and v_next > threshold and v_bridge < threshold:
            drop.append(i)

    return x.drop(index=drop).reset_index(drop=True), drop


state_clean, dropped_state_rows = remove_isolated_state_outliers(
    cleaned["state"], joint_cols, speed_limit
)
cleaned["state"] = state_clean
print("Dropped isolated state rows:", dropped_state_rows)

## 25. Synchronize modalities on a visual timebase

We use front-camera timestamps, nearest-match the wrist camera, reject poor camera sync, then interpolate continuous state/action values.

In [ ]:
def interpolate_numeric(df, target_t, columns):
    src_t = df["t"].to_numpy(float)
    out = {}
    for col in columns:
        out[col] = np.interp(target_t, src_t, df[col].to_numpy(float))
    return pd.DataFrame(out)


front = cleaned["front"]
wrist = cleaned["wrist"]
action = cleaned["action"]
state = cleaned["state"]

start_t = max(front.t.min(), wrist.t.min(), action.t.min(), state.t.min())
end_t = min(front.t.max(), wrist.t.max(), action.t.max(), state.t.max())

front_use = front[(front.t >= start_t) & (front.t <= end_t)].copy().reset_index(drop=True)

wrist_t = wrist["t"].to_numpy(float)
front_t = front_use["t"].to_numpy(float)
nearest_idx = np.abs(wrist_t[:, None] - front_t[None, :]).argmin(axis=0)
nearest_wrist_t = wrist_t[nearest_idx]
cam_sync_error = np.abs(nearest_wrist_t - front_t)

nominal_cam_dt = np.median(np.diff(np.sort(front_t)))
MAX_CAMERA_SYNC_ERROR = 0.5 * nominal_cam_dt
keep = cam_sync_error <= MAX_CAMERA_SYNC_ERROR

timeline = front_use.loc[keep, "t"].to_numpy(float)
matched_wrist = wrist.iloc[nearest_idx[keep]].reset_index(drop=True)

state_cols = [c for c in state.columns if c != "t"]
action_cols = [c for c in action.columns if c != "t"]

state_aligned = interpolate_numeric(state, timeline, state_cols)
action_aligned = interpolate_numeric(action, timeline, action_cols)

aligned = pd.DataFrame(
    {
        "t": timeline,
        "front_frame_id": front_use.loc[keep, "frame_id"].to_numpy(),
        "wrist_frame_id": matched_wrist["frame_id"].to_numpy(),
        "camera_sync_error_s": cam_sync_error[keep],
    }
)
for c in state_cols:
    aligned[f"state.{c}"] = state_aligned[c].to_numpy()
for c in action_cols:
    aligned[f"action.{c}"] = action_aligned[c].to_numpy()

print("Frames before camera-sync filter:", len(front_use))
print("Frames after  camera-sync filter:", len(aligned))
print("Max accepted camera sync error:", aligned.camera_sync_error_s.max())
display(aligned.head())

## 26. Visualize raw → cleaned → aligned data

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(streams["state"]["t"], streams["state"]["arm_0"], ".", alpha=0.4, label="raw state arm_0")
plt.plot(cleaned["state"]["t"], cleaned["state"]["arm_0"], "-", label="cleaned state arm_0")
plt.plot(aligned["t"], aligned["state.arm_0"], "o", markersize=3, label="aligned @ camera time")
plt.xlabel("absolute time (s)")
plt.ylabel("joint/state value")
plt.title("Raw → cleaned → camera-aligned")
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(aligned["t"], aligned["camera_sync_error_s"] * 1000)
plt.xlabel("absolute time (s)")
plt.ylabel("front↔wrist timestamp error (ms)")
plt.title("Camera synchronization error")
plt.show()

## 27. Head/tail idle trimming

This uses action motion as a simple first pass. Production pipelines should combine it with task-state/success annotations.

In [ ]:
a_cols = [c for c in aligned.columns if c.startswith("action.arm_")]
a = aligned[a_cols].to_numpy(float)
t = aligned["t"].to_numpy(float)

if len(aligned) >= 3:
    da = np.linalg.norm(np.diff(a, axis=0), axis=1)
    dt = np.diff(t)
    action_speed = da / np.maximum(dt, 1e-9)
    move_threshold = max(np.percentile(action_speed, 20), 1e-5)
    moving = np.r_[False, action_speed > move_threshold]

    idx = np.where(moving)[0]
    if len(idx):
        margin_frames = 3
        lo = max(0, idx[0] - margin_frames)
        hi = min(len(aligned), idx[-1] + margin_frames + 1)
        aligned_trimmed = aligned.iloc[lo:hi].reset_index(drop=True)
    else:
        aligned_trimmed = aligned.copy()
else:
    aligned_trimmed = aligned.copy()

print("Before trim:", len(aligned))
print("After trim :", len(aligned_trimmed))

## 28. Export a GR00T-shaped staging episode

The export mirrors the SO100 5-arm+gripper dimensional convention and creates tiny synthetic MP4s only so the final directory shape is inspectable.

**The generated `info.json` is staging metadata, not a complete production LeRobot export.** For real training, use real images and regenerate/validate complete LeRobot metadata and GR00T statistics.

In [ ]:
import shutil

import imageio.v2 as imageio
from PIL import Image, ImageDraw

EXPORT_ROOT = WORK_ROOT / "mcap" / "cleaned_episode_gr00t_staging"
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)

meta_dir = EXPORT_ROOT / "meta"
data_dir = EXPORT_ROOT / "data" / "chunk-000"
front_dir = EXPORT_ROOT / "videos" / "chunk-000" / "observation.images.front"
wrist_dir = EXPORT_ROOT / "videos" / "chunk-000" / "observation.images.wrist"

for d in [meta_dir, data_dir, front_dir, wrist_dir]:
    d.mkdir(parents=True, exist_ok=True)

df = aligned_trimmed.copy()
t0 = df["t"].iloc[0]
rel_t = df["t"].to_numpy(float) - t0

state_cols_export = [f"state.arm_{i}" for i in range(5)] + ["state.gripper"]
action_cols_export = [f"action.arm_{i}" for i in range(5)] + ["action.gripper"]

episode_df = pd.DataFrame(
    {
        "observation.state": [
            np.asarray(row, dtype=np.float32) for row in df[state_cols_export].to_numpy()
        ],
        "action": [np.asarray(row, dtype=np.float32) for row in df[action_cols_export].to_numpy()],
        "timestamp": rel_t.astype(np.float64),
        "annotation.human.task_description": np.zeros(len(df), dtype=np.int64),
        "task_index": np.zeros(len(df), dtype=np.int64),
        "episode_index": np.zeros(len(df), dtype=np.int64),
        "index": np.arange(len(df), dtype=np.int64),
        "next.reward": np.zeros(len(df), dtype=np.float32),
        "next.done": np.r_[np.zeros(max(0, len(df) - 1), dtype=bool), True],
    }
)
episode_df.to_parquet(data_dir / "episode_000000.parquet", index=False)

modality_json = {
    "state": {
        "single_arm": {"start": 0, "end": 5},
        "gripper": {"start": 5, "end": 6},
    },
    "action": {
        "single_arm": {"start": 0, "end": 5},
        "gripper": {"start": 5, "end": 6},
    },
    "video": {
        "front": {"original_key": "observation.images.front"},
        "wrist": {"original_key": "observation.images.wrist"},
    },
    "annotation": {
        "human.task_description": {"original_key": "task_index"},
    },
}
(meta_dir / "modality.json").write_text(json.dumps(modality_json, indent=2))

task = "pick the red cube and place it in the bowl"
(meta_dir / "tasks.jsonl").write_text(json.dumps({"task_index": 0, "task": task}) + "\n")
(meta_dir / "episodes.jsonl").write_text(
    json.dumps(
        {
            "episode_index": 0,
            "tasks": [task],
            "length": len(episode_df),
        }
    )
    + "\n"
)

fps = 1.0 / np.median(np.diff(rel_t)) if len(rel_t) > 1 else 15.0
info_staging = {
    "codebase_version": "staging-created-by-gr00t-lab",
    "robot_type": "industrial_arm_demo",
    "total_episodes": 1,
    "total_frames": len(episode_df),
    "fps": float(fps),
    "note": "Staging metadata only. Regenerate/validate with official LeRobot/GR00T tooling before training.",
}
(meta_dir / "info.json").write_text(json.dumps(info_staging, indent=2))

video_fps = max(1, round(fps))


def make_frame(i, n, label, wrist=False):
    img = Image.new("RGB", (320, 180), (245, 245, 245))
    d = ImageDraw.Draw(img)
    frac = i / max(1, n - 1)
    x = int(40 + frac * 220)
    y = int(90 + (30 if wrist else 45) * math.sin(frac * 2 * math.pi))
    d.ellipse((x - 10, y - 10, x + 10, y + 10), fill=(60, 80, 180))
    d.text((10, 10), f"{label} frame {i}", fill=(0, 0, 0))
    return np.asarray(img)


for path, label, wrist_flag in [
    (front_dir / "episode_000000.mp4", "front", False),
    (wrist_dir / "episode_000000.mp4", "wrist", True),
]:
    writer = imageio.get_writer(str(path), fps=video_fps, codec="libx264", quality=7)
    for i in range(len(episode_df)):
        writer.append_data(make_frame(i, len(episode_df), label, wrist=wrist_flag))
    writer.close()

print("Exported staging dataset:", EXPORT_ROOT)
for p in sorted(EXPORT_ROOT.rglob("*")):
    if p.is_file():
        print(p.relative_to(EXPORT_ROOT))

## 29. Audit the cleaned staging episode

In [ ]:
staging_audit = audit_gr00t_dataset(EXPORT_ROOT)
display(staging_audit)

display(pd.read_parquet(EXPORT_ROOT / "data/chunk-000/episode_000000.parquet").head())

## 30. Generate GR00T statistics before real training

Once a real converted dataset has complete LeRobot v2 metadata, compute GR00T statistics:

```bash
uv run python gr00t/data/stats.py   --dataset-path /path/to/your_dataset   --embodiment-tag NEW_EMBODIMENT   --modality-config-path examples/SO100/so100_config.py
```

If your industrial arm has a different state/action space, create a matching modality config instead of reusing SO100.

# Part E — translate this to an industrial arm

A practical architecture:

```text
                       task instruction
                              │
                              ▼
front camera ────────────── GR00T
wrist camera ────────────────┤
joint state / EEF pose ──────┤
                              │
                       future action chunk
                              │
                              ▼
                    safety / rate limiter
                              │
                              ▼
                    robot controller / ROS2
                              │
                        physical arm
```

For a UR5e / ABB / FANUC-style experiment:

1. choose joint-space or Cartesian EEF control;
2. record cameras, state, demonstrated target, gripper, task text and optionally force/torque;
3. prefer one MCAP per episode;
4. QA timestamps, camera sync, holes, physical limits, stale frames, task success and idle periods;
5. convert to GR00T LeRobot;
6. write a custom `modality.json` and modality config;
7. generate statistics;
8. start with a narrow workspace/distribution;
9. fine-tune;
10. do open-loop checks, then supervised closed-loop evaluation.

### Safety boundary

Do not connect a freshly fine-tuned policy directly to an industrial robot at unrestricted speed. Keep normal safety envelopes, conservative rate/position limits, an emergency stop and supervised low-speed evaluation.

# Part F — suggested learning sequence

1. **Understand the data:** run sections 5–10.
2. **Prove fine-tuning works:** run the 50-step SO100 smoke test.
3. **Richer manipulation:** inspect LIBERO Goal, then fine-tune briefly or use NVIDIA's checkpoint.
4. **Data engineering:** run the MCAP exercise and deliberately inject larger gaps, stale frames, clock offsets and gripper discontinuities.
5. **Bring your own ROS 2 episode:** point `MCAP_PATH` at it, inventory topics, inspect schemas, then replace `decode_demo_mcap()` with your real mappings.

# Runtime matrix

| Section | macOS | Linux CPU | Linux/Docker + NVIDIA GPU |
|---|---:|---:|---:|
| Clone repo / inspect SO100 | ✅ | ✅ | ✅ |
| LeRobot parquet/video QA | ✅ | ✅ | ✅ |
| MuJoCo fundamentals | ✅ | ✅ | ✅ |
| MCAP cleaning | ✅ | ✅ | ✅ |
| LIBERO dataset download/inspection | ✅ | ✅ | ✅ |
| GR00T N1.7 fine-tuning | — | — | ✅ |
| GR00T open-loop model inference | — | — | ✅ |
| LIBERO + GR00T closed-loop MuJoCo | — | — | ✅ |
| Before/after rollout videos | — | — | ✅ |

`—` means the notebook intentionally skips that path rather than treating the platform as broken.

# Appendix — public downloads

| Purpose | Resource |
|---|---|
| GR00T source + bundled demos | `https://github.com/NVIDIA/Isaac-GR00T` |
| GR00T N1.7 base | `nvidia/GR00T-N1.7-3B` |
| Required VLM backbone | `nvidia/Cosmos-Reason2-2B` |
| LIBERO Goal data | `IPEC-COMMUNITY/libero_goal_no_noops_1.0.0_lerobot` |
| LIBERO Object data | `IPEC-COMMUNITY/libero_object_no_noops_1.0.0_lerobot` |
| LIBERO Spatial data | `IPEC-COMMUNITY/libero_spatial_no_noops_1.0.0_lerobot` |
| LIBERO-10 data | `IPEC-COMMUNITY/libero_10_no_noops_1.0.0_lerobot` |
| NVIDIA LIBERO checkpoints | `nvidia/GR00T-N1.7-LIBERO` |
| DROID sample generator | bundled `scripts/download_droid_sample.py` |
| Full DROID source | `lerobot/droid_1.0.1` |
| MCAP Python | `mcap` |
| ROS 2 MCAP decoding without ROS | `mcap-ros2-support` |

The full DROID dataset is very large; NVIDIA's example script can fetch a small sample for pipeline testing instead.

| MuJoCo Python runtime/docs | `https://mujoco.readthedocs.io/en/stable/python.html` |
| LIBERO benchmark | `https://github.com/Lifelong-Robot-Learning/LIBERO` |
